# კვირა IV: სრული ML ციკლი ერთ ამოცანაზე (The Full ML Pipeline)
**კურსი:** ხელოვნური ინტელექტი და მანქანური სწავლების საწყისები (BTU)  
**სახელმძღვანელო:** Chris Albon, *Machine Learning with Python Cookbook* (1st Edition, O'Reilly)  
**მონაცემები:** Adult Census Income (აშშ-ის მოსახლეობის აღწერა). ამოცანა: გამოიმუშავებს თუ არა ადამიანი წელიწადში **$50,000-ზე მეტს**?

---
### 🎯 ლექციის მიზანი და არქიტექტურა:
> ამ Notebook-ში აწყობილია **ერთი სრული, უსაფრთხო Pipeline ობიექტი**, რომელიც ნედლ, რეალურ მონაცემებს (ტექსტური სვეტები, გამოტოვებული უჯრები, განსხვავებული მასშტაბები) პირდაპირ გარდაქმნის, ამცირებს განზომილებას PCA-ით, არჩევს საუკეთესო ჰიპერპარამეტრებს `GridSearchCV`-ით და აფასებს საბოლოო შედეგს სატესტო შერჩევაზე.

### 🗺️ სრული ციკლის რუკა:
```
ნედლი ცხრილი ──► [Train/Test Split] ──► ColumnTransformer ──► PCA ──► kNN ──► შეფასება (F1 & CM)
                     (გაჟონვის პრევენცია)     (Imputer + OneHot + Scaler)
```


### 1. საბაზისო ბიბლიოთეკები
დავიწყოთ მონაცემთა დამუშავების საბაზისო ხელსაწყოებით (Pandas, NumPy, Matplotlib) და Scikit-Learn-ის OpenML ჩამტვირთველით.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

print("საბაზისო ბიბლიოთეკები მზადაა!")


### 2. მონაცემების ჩატვირთვა და სვეტების ავტომატური დახარისხება
მონაცემები პირდაპირ იტვირთება OpenML-იდან (`fetch_openml`) — სრული 48,842 ჩანაწერი.  
*(შენიშვნა Kaggle-ის მომხმარებლებისთვის: დარწმუნდით, რომ ჩართულია **Settings -> Internet ON**).*

ჩვენ ვაკეთებთ ორ რამეს:
1. **ზედმეტი სვეტების მოცილება:**
   * `education`: 100%-ით დუბლირებს `education-num`-ს (რომელიც უკვე რიცხვითია: 1-16). მისი დატოვება OneHot-ით 16 ზედმეტ სვეტს გააჩენდა და რიცხვით იერარქიას დააკარგვინებდა.
   * `fnlwgt`: აღწერის ბიუროს სტატისტიკური შერჩევის წონაა (რამდენ მოქალაქეს წარმოადგენს ეს პირი). ის ადამიანის პირად შემოსავალს არ განსაზღვრავს და მოდელისთვის სუფთა ხმაურია.
   * `class`: სამიზნე ცვლადია ($y$), ამიტომ მას $X$-იდან ვაგდებთ.
2. **სვეტების ავტომატური დახარისხება (`select_dtypes`):** Pandas-ი თავად პოულობს რიცხვით და კატეგორიულ სვეტებს — სახელების ხელით ჩამოწერის გარეშე.


In [ ]:
df = fetch_openml("adult", version=2, as_frame=True).frame

# გამოვყოთ X და y სრული მონაცემებიდან (48,842 ჩანაწერი):
# 'class' არის სამიზნე ცვლადი (1 = გამოიმუშავებს >50K-ს, 0 = <=50K)
X = df.drop(columns=["fnlwgt", "education", "class"])
y = (df["class"] == ">50K").astype(int)

# სვეტების ავტომატური დახარისხება ტიპების მიხედვით:
NUMERIC_COLS = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL_COLS = X.select_dtypes(exclude="number").columns.tolist()

print(f"მონაცემების ზომა: {X.shape}")
print(f"კლასების ბალანსი: {y.mean():.1%} გამოიმუშავებს >50K-ს")
print(f"\nრიცხვითი სვეტები ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"კატეგორიული სვეტები ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")
X.head()


## 3. ჯერ ვყოფთ — Train / Test Split (Recipe 11.1)
**ოქროს წესი:** სატესტო ნაწილი არის მომავალი. მონაცემთა მომზადება (საშუალო, მედიანა, PCA ღერძები) უნდა ვისწავლოთ **მხოლოდ** სასწავლო მონაცემებზე, რათა ავირიდოთ მონაცემთა გაჟონვა (Data Leakage).

ვიყენებთ `train_test_split`-ს `stratify=y`-ით (კლასების პროპორციის შესანარჩუნებლად) და `StratifiedKFold`-ს კროს-ვალიდაციისთვის.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 5-fold კროს-ვალიდაციის სტრატეგია (მუდმივი ფოლდები სამართლიანი შედარებისთვის):
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("სატესტო ნაწილი დაიკეტა საბოლოო შეფასებამდე!")


## 4. მომზადების კონვეიერი: `ColumnTransformer` (Recipe 12.3)
ჩვენს ცხრილში გვაქვს ორი ტიპის სვეტი:
1. **რიცხვითი სვეტები:** სჭირდება `SimpleImputer(strategy='median')` და `StandardScaler()`
2. **კატეგორიული სვეტები:** სჭირდება `SimpleImputer(strategy='most_frequent')` და `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`

* `sparse_output=False`: აბრუნებს მკვრივ (dense) NumPy მასივს, რაც აუცილებელია PCA-სთვის.
* `PCA(n_components=n_components)`: თუ `with_pca=False`, ჩაენაცვლება `"passthrough"` (მონაცემები გაივლის შეუცვლელად).


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

def build_pipeline(n_components=10, k=15, with_pca=True):
    """ქმნის ერთიან Scikit-Learn მილსადენს."""
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipe, NUMERIC_COLS),
        ("cat", cat_pipe, CATEGORICAL_COLS)
    ])

    pca_step = PCA(n_components=n_components, random_state=42) if with_pca else "passthrough"

    return Pipeline([ 
        ("prep", preprocessor),
        ("pca", pca_step),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

print("ფუნქცია build_pipeline მზადაა!")


## 5. მოდელის შეფასება 5-Fold კროს-ვალიდაციით (Recipe 11.1)
სრულ მილსადენს ვაფასებთ 5-Fold კროს-ვალიდაციით `StratifiedKFold`-ზე და ვზომავთ გაშვების დროს. მეტრიკად ვიყენებთ $F_1$-ქულას, რადგან სამიზნე კლასები არაბალანსირებულია.


In [ ]:
import time
from sklearn.model_selection import cross_val_score

pipe = build_pipeline(n_components=10, k=15)
t0 = time.perf_counter()
f1_full = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1").mean()
t_full = time.perf_counter() - t0

print(f"5-Fold CV F1 Score (სრული Pipeline, 10 კომპონენტი): {f1_full:.3f} | დრო: {t_full:.2f} წმ")


## 6. Ablation Study: PCA-ს ეფექტი (სისწრაფე vs ინფორმაცია)
მანქანურ სწავლებაში **Ablation Study** ჰქვია რომელიმე კომპონენტის გათიშვას და ეფექტის შეფასებას:
* **PCA-ს გარეშე (`with_pca=False`):** მონაცემთა მომზადების შემდეგ (სკალირებული და One-Hot) მოდელი პირდაპირ მიიღებს ყველა 80+ სვეტს.
* შევადაროთ 10 კომპონენტიანი PCA-ს და ყველა სვეტის შედეგი: რა მოუვა ხარისხსა ($F_1$) და გამოთვლის დროს?


In [ ]:
# შედარება: რა მოხდება PCA-ს გარეშე (ყველა სვეტით kNN-ში)?
pipe_no_pca = build_pipeline(with_pca=False)
t0 = time.perf_counter()
f1_no_pca = cross_val_score(pipe_no_pca, X_train, y_train, cv=cv, scoring="f1").mean()
t_no_pca = time.perf_counter() - t0

print(f"5-Fold CV F1 Score (PCA-ს გარეშე, ყველა სვეტი): {f1_no_pca:.3f} | დრო: {t_no_pca:.2f} წმ")
speedup = t_no_pca / t_full if t_full > 0 else 1.0
print(f"-> კომპრომისი (Trade-off): 10 კომპონენტიანი PCA {speedup:.1f}x-ით უფრო სწრაფია,")
print(f"   ხოლო F1 ქულის სხვაობა მინიმალურია ({f1_full:.3f} vs {f1_no_pca:.3f})!")


## 7. PCA: ახსნილი დისპერსია და Scree Plot / Elbow Curve (Recipe 9.1)
რამდენ ინფორმაციას ინარჩუნებს PCA შეკუმშვის შემდეგ და როგორ ავარჩიოთ კომპონენტების ოპტიმალური რაოდენობა?
* **Scree Plot (Elbow Curve):** ვხატავთ, როგორ იზრდება ახსნილი დისპერსიის კუმულაციური წილი კომპონენტების რაოდენობის მატებასთან ერთად.
* ვაკვირდებით ე.წ. „იდაყვს“ (Elbow) — წერტილს, რომლის შემდეგაც ყოველი ახალი კომპონენტი სულ უფრო ცოტა დამატებით ინფორმაციას იძლევა.


In [ ]:
# პირდაპირ ვქმნით და ვწვრთნით მილსადენს 25 კომპონენტით იდაყვის (Elbow) სანახავად:
pipe_scree = build_pipeline(n_components=25)
pipe_scree.fit(X_train, y_train)

pca = pipe_scree.named_steps["pca"]
var_ind = pca.explained_variance_ratio_ * 100
var_cum = np.cumsum(var_ind)

print(f"10 კომპონენტის მიერ ახსნილი ჯამური დისპერსია: {var_cum[9]:.1f}%")
print(f"25 კომპონენტის მიერ ახსნილი ჯამური დისპერსია: {var_cum[-1]:.1f}%\n")

# Scree Plot (Elbow Curve) ვიზუალიზაცია:
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, 26), var_cum, marker="o", color="#1f77b4", linewidth=2, label="კუმულაციური დისპერსია (%)")
plt.bar(range(1, 26), var_ind, alpha=0.35, color="#ff7f0e", label="ინდივიდუალური წილი (თითო კომპონენტი)")
plt.axvline(x=10, color="r", linestyle="--", alpha=0.8, label=f"ჩვენი არჩევანი: 10 კომპონენტი ({var_cum[9]:.1f}%)")
plt.title("PCA Scree Plot: ახსნილი დისპერსიის ზრდა (Elbow Curve)")
plt.xlabel("კომპონენტების რაოდენობა")
plt.ylabel("ახსნილი დისპერსია (%)")
plt.xticks(range(1, 26, 2))
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


## 8. ჰიპერპარამეტრების შერჩევა: `GridSearchCV` (Recipe 12.1, 12.3)
რამდენი კომპონენტია ოპტიმალური PCA-სთვის? რომელი $k$ არის საუკეთესო?  
წინასწარ მარჩიელობის ნაცვლად, ვიყენებთ `GridSearchCV`-ს.

* **სინტაქსის წესი:** Pipeline-ში პარამეტრის მითითებისას ვწერთ: `ნაბიჯის_სახელი` + `__` (ორი ქვედა ტირე) + `პარამეტრი` (მაგ. `pca__n_components` და `knn__n_neighbors`).
* `n_jobs=-1`: იყენებს პროცესორის ყველა ბირთვს პარალელური გამოთვლებისთვის.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "pca__n_components": [5, 10, 20, 25],
    "knn__n_neighbors": [5, 10, 15, 20, 25]
}

grid = GridSearchCV(build_pipeline(), param_grid, cv=cv, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

print(f"საუკეთესო პარამეტრები: {grid.best_params_}")
print(f"საუკეთესო CV F1: {grid.best_score_:.3f}")


## 9. საბოლოო გამოცდა: სატესტო შერჩევის შეფასება (Recipe 11.3, 11.6)
სატესტო მონაცემებს (`X_test`, `y_test`) ვხსნით მხოლოდ ახლა — ზუსტად ერთხელ.  
`grid.predict` ავტომატურად იყენებს საუკეთესო პარამეტრებით გაწვრთნილ მოდელს. ვაგენერირებთ `classification_report`-სა და აღრევის მატრიცას (`Confusion Matrix`).


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = grid.predict(X_test)

print("--- სატესტო შედეგების რეპორტი ---")
print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["<=50K", ">50K"]).plot(cmap="Blues")
plt.title("Confusion Matrix (სატესტო მონაცემები)")
plt.tight_layout()
plt.show()


---
## 🎫 შეჯამება და ძირითადი დასკვნები
1. **მონაცემთა გაჟონვის თავიდან აცილება:** მონაცემები ჯერ გავყავით `train/test`-ად და მხოლოდ სასწავლო მონაცემებზე ვისწავლეთ იმპუტაცია, სკალირება და PCA.
2. **ColumnTransformer + Pipeline:** სხვადასხვა ტიპის მონაცემების (რიცხვითი და კატეგორიული) დამუშავება და მოდელირება გაერთიანდა ერთ დახურულ ობიექტში.
3. **PCA სისწრაფე vs ხარისხი:** PCA-მ 80+ განზომილებიდან მონაცემები შეკუმშა 10 კომპონენტამდე, რამაც გამოთვლები საგრძნობლად დააჩქარა $F_1$ ხარისხის თითქმის უცვლელად შენარჩუნებით.
4. **GridSearchCV:** ოპტიმალური ჰიპერპარამეტრები ერთიანად შეირჩა კროს-ვალიდაციით.
